In [37]:
%%time
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import wandb


# Table processing
def process_line(means, highlight, highlight_index, highlight_max, ignore_std):
    if highlight:
        if highlight_max:
            tops = set(means.groupby(highlight_index).idxmax())
        else:
            tops = set(means.groupby(highlight_index).idxmin())
    else:
        tops = set()

    def process_line(x):
        if ignore_std:
            if x.name in tops:
                return rf"\textbf{{{x['mean']:0.3f}}}"
            return rf"{x['mean']:0.3f}"
        if x.name in tops:
            return rf"\textbf{{{x['mean']:0.3f} $\pm$ {x['std']:0.3f}}}"
        return rf"{x['mean']:0.3f} $\pm$ {x['std']:0.3f}"

    return process_line


def mean_pm_std(
    data,
    index,
    columns,
    value,
    highlight=True,
    highlight_cols=True,
    highlight_max=True,
    ignore_std=False,
):
    assert len(data) > 0
    groupby = data.groupby([*index, *columns])
    means = groupby.mean()[value].rename("mean")
    stds = groupby.std()[value].rename("std")
    ddf = pd.concat([means, stds], axis=1).T
    highlight_index = columns if highlight_cols else index
    ddf = ddf.apply(
        process_line(means, highlight, highlight_index, highlight_max, ignore_std)
    )
    ddf = ddf.reset_index().pivot(index=index, columns=columns)
    ddf.columns = ddf.columns.droplevel(level=0)
    return ddf

    
def flatten_dict(d, parent_key="", sep="/"):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)


def prepare_data(data):
    flattened_data = [flatten_dict(item) for item in data]
    return pd.DataFrame(flattened_data)



The jupyter_black extension is already loaded. To reload it, use:
  %reload_ext jupyter_black
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
CPU times: user 527 μs, sys: 376 μs, total: 903 μs
Wall time: 1.01 ms


In [38]:
api = wandb.Api(timeout=30)

# Project is specified by <entity/project-name>
runs = api.runs(
    "openproblems-comp/fast-tbg",
    filters={
        "$and": [
            {
                # "created_at": {"$gt": "2025-01-26T00:00:00"},
                "tags": {"$in": ["jarz_big_v3", "jarz_final_iv10"]},
                #'group': {'$in': ['5_vars']},
                # "config.data.n_particles": {"$eq": 22},
                #'config.model': {'$eq': model},
                #'config.lr': {'$lt': 1.01 * lr, '$gt': 0.99 * lr},
            }
        ]
    },
)

summary_list, config_list, name_list, tag_list = [], [], [], []
for run in runs:
    tag_list.append(run.tags)
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)
    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
    # .name is the human-readable name of the run.
    name_list.append(run.name)
df_summary = prepare_data(summary_list)
df_config = prepare_data(config_list)
tag_list = [str(t) for t in tag_list]
df = pd.concat(
    [
        pd.DataFrame(name_list, columns=["name"]),
        pd.DataFrame(tag_list, columns=["Tags"]),
        df_summary,
        df_config,
    ],
    axis=1,
)

In [39]:
metrics = [
    "test/jarzynski/energy_w1",
    "test/jarzynski/effective_sample_size",
    "test/jarzynski/rama/torus_wasserstein",
]
# metrics = [
#     "test/cropped_energy_w1",
#     "test/effective_sample_size",
#     "test/rama/torus_wasserstein",
# ]
# metrics = [
#     "test/resampled/energy_w1",
#     "test/effective_sample_size",
#     "test/resampled/rama/torus_wasserstein",
# ]

In [40]:
import math


def filterer(x):
    if isinstance(x, float) and not math.isfinite(x):
        return False
    return "table" in list(x)


filtered_df = df[
    # df["model/_target_"].isin(["src.models.flow_matching_module.FlowMatchLitModule"])
    # & df["Tags"].apply(lambda x: "eval" in x)
    df["Tags"].apply(lambda x: True)
    # & ~df["val/cropped_energy_w1"].isna()
    # & df["tags"].apply(filterer)
][
    [
        # "tags",
        "model/_target_",
        "model/net/_target_",
        "data/n_particles",
        "model/use_com_energy",
        "model/sampling_config/num_test_proposal_samples",
        *metrics,
    ]
].drop(
    columns=["model/_target_"]
)

filtered_df = filtered_df[filtered_df["model/use_com_energy"] is True]
filtered_df = filtered_df[
    filtered_df["model/sampling_config/num_test_proposal_samples"] == 100_000
]

# filtered_df.sort_values("data/n_particles")

In [41]:
renamed_df = filtered_df.replace(
    {
        "src.models.components.tbg.egnn_dynamics_ad2_cat.EGNN_dynamics_AD2_cat": "EQ-CFM",
        "src.models.components.dit.DIT3D": "DiT-CFM",
        "src.models.components.tarflow.TarFlow": "TarFlow",
    }
).rename(
    columns={
        "model/net/_target_": "Model",
        "data/n_particles": "n_particles",
        "test/cropped_energy_w1": "test/fcropped_energy_w1",
    }
)  # so order is correct lol

if "test/cropped_energy_w1" in metrics:
    metrics[metrics.index("test/cropped_energy_w1")] = "test/fcropped_energy_w1"

In [42]:
renamed_df.groupby(["Model", "n_particles"]).count()

model/use_com_energy  \
Model   n_particles                         
TarFlow 22                              3   
        33                              3   
        42                              3   
        63                              1   

                     model/sampling_config/num_test_proposal_samples  \
Model   n_particles                                                    
TarFlow 22                                                         3   
        33                                                         3   
        42                                                         3   
        63                                                         1   

                     test/jarzynski/energy_w1  \
Model   n_particles                             
TarFlow 22                                  3   
        33                                  3   
        42                                  3   
        63                                  1   

                     test/jarzynski/effective_sample_size  \
Model   n_particles                                         
TarFlow 22                                              3   
        33                                              3   
        42                                              3   
        63                                              1   

                     test/jarzynski/rama/torus_wasserstein  
Model   n_particles                                         
TarFlow 22                                               3  
        33                                               3  
        42                                               3  
        63                                               1

In [43]:
df_melt = renamed_df.dropna().melt(
    value_vars=metrics, id_vars=["Model", "n_particles"], var_name="Metric"
)

In [44]:
results = mean_pm_std(
    df_melt, index=["Model"], columns=["n_particles", "Metric"], value="value", highlight=False
)
results

n_particles                                   22                           \
Metric      test/jarzynski/effective_sample_size test/jarzynski/energy_w1   
Model                                                                       
TarFlow                        0.894 $\pm$ 0.172        0.466 $\pm$ 0.380   

n_particles                                        \
Metric      test/jarzynski/rama/torus_wasserstein   
Model                                               
TarFlow                         0.226 $\pm$ 0.029   

n_particles                                   33                           \
Metric      test/jarzynski/effective_sample_size test/jarzynski/energy_w1   
Model                                                                       
TarFlow                        0.907 $\pm$ 0.151        1.314 $\pm$ 0.409   

n_particles                                        \
Metric      test/jarzynski/rama/torus_wasserstein   
Model                                               
TarFlow                         0.948 $\pm$ 0.068   

n_particles                                   42                           \
Metric      test/jarzynski/effective_sample_size test/jarzynski/energy_w1   
Model                                                                       
TarFlow                        0.886 $\pm$ 0.087        1.920 $\pm$ 0.251   

n_particles                                        \
Metric      test/jarzynski/rama/torus_wasserstein   
Model                                               
TarFlow                         1.841 $\pm$ 0.047   

n_particles                                   63                           \
Metric      test/jarzynski/effective_sample_size test/jarzynski/energy_w1   
Model                                                                       
TarFlow                          0.989 $\pm$ nan          0.305 $\pm$ nan   

n_particles                                        
Metric      test/jarzynski/rama/torus_wasserstein  
Model                                              
TarFlow                           3.326 $\pm$ nan

In [45]:
print(
    results.to_latex(
        float_format="{:.3f}".format,
    )
)

\begin{tabular}{lllllllllllll}
\toprule
n_particles & \multicolumn{3}{r}{22} & \multicolumn{3}{r}{33} & \multicolumn{3}{r}{42} & \multicolumn{3}{r}{63} \\
Metric & test/jarzynski/effective_sample_size & test/jarzynski/energy_w1 & test/jarzynski/rama/torus_wasserstein & test/jarzynski/effective_sample_size & test/jarzynski/energy_w1 & test/jarzynski/rama/torus_wasserstein & test/jarzynski/effective_sample_size & test/jarzynski/energy_w1 & test/jarzynski/rama/torus_wasserstein & test/jarzynski/effective_sample_size & test/jarzynski/energy_w1 & test/jarzynski/rama/torus_wasserstein \\
Model &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
TarFlow & 0.894 $\pm$ 0.172 & 0.466 $\pm$ 0.380 & 0.226 $\pm$ 0.029 & 0.907 $\pm$ 0.151 & 1.314 $\pm$ 0.409 & 0.948 $\pm$ 0.068 & 0.886 $\pm$ 0.087 & 1.920 $\pm$ 0.251 & 1.841 $\pm$ 0.047 & 0.989 $\pm$ nan & 0.305 $\pm$ nan & 3.326 $\pm$ nan \\
\bottomrule
\end{tabular}

